In [ ]:
from src.gaussian_mixture import GaussianMixtureMAR
from src.motivating_example import MotivatingExample
from src.data_generator import DataGenerator
import numpy as np

import matplotlib.pyplot as plt

import torch
import rpy2.robjects as ro
from rpy2.robjects.packages import importr

In [10]:


mice = importr("mice")

class AmputeMAR:

    def _to_r_dataframe(self, X):
        return ro.DataFrame({
            f"X{j + 1}": ro.FloatVector(X[:, j])
            for j in range(X.shape[1])
        })

    def _to_r_matrix(self, X):
        return ro.r.matrix(
            ro.IntVector(X.T.flatten().tolist()),
            nrow=X.shape[0],
            ncol=X.shape[1],
        )

    @staticmethod
    def sample_patterns(d, n, rng=None):
        """Sample incomplete missingness patterns once (1 = observed, 0 = missing).
        Last column is always observed; all-ones rows are excluded."""
        rng = np.random.default_rng() if rng is None else rng
        while True:
            npattern = int(rng.integers(3, max(round(n / 100), 3) + 1))
            patterns = rng.integers(0, 2, size=(npattern, d - 1))
            patterns = np.hstack([patterns, np.ones((npattern, 1), dtype=int)])
            incomplete = patterns[~patterns.all(axis=1)]
            if incomplete.shape[0] > 0:
                return incomplete

    def ampute(self, X, patterns=None, prop=None, rng=None):
        """
        Parameters
        ----------
        X : torch.Tensor, shape (n, d)
            Complete data.
        patterns : array-like, shape (n_pats, d), optional
            Binary patterns: 1 = observed, 0 = missing. All-ones rows are dropped.
            If None, new patterns are sampled.
        prop : float, optional
            Proportion of cases with missing values (so 1 - prop stay complete).
            Default: n_pats / (n_pats + 1), i.e. the complete cases are as
            frequent as each incomplete pattern.
        rng : np.random.Generator, optional
            Only used if patterns have to be sampled.

        Returns
        -------
        X_NA : torch.Tensor
            Data with NaN values.
        M : torch.Tensor
            Observation mask: 1 = observed, 0 = missing.
        """
        n, d = X.shape
        device = X.device
        X_np = X.detach().cpu().numpy().astype(float)

        if patterns is None:
            patterns = self.sample_patterns(d, n, rng)
        patterns = np.asarray(patterns, dtype=int)
        patterns = patterns[~patterns.all(axis=1)]
        n_pats = patterns.shape[0]

        if prop is None:
            prop = n_pats / (n_pats + 1)

        X_NA_np = X_np.copy()

        if n_pats > 0:
            result = mice.ampute(
                self._to_r_dataframe(X_np),
                patterns=self._to_r_matrix(patterns),
                prop=prop,              # freq defaults to equal across patterns
                mech="MAR",
                bycases=True,
            )
            X_NA_np = np.asarray(result.rx2("amp")).T   # (n, d)

        M_np = (~np.isnan(X_NA_np)).astype(np.float32)

        return (
            torch.tensor(X_NA_np, dtype=X.dtype, device=device),
            torch.tensor(M_np, dtype=X.dtype, device=device),
        )





In [16]:
class ParametricEstimation(GaussianMixtureMAR, AmputeMAR):

    def __init__(self,
                 data_generator,
                 n,
                 d,
                 patterns,
                 k_range,
                 criterion="bic",
                 cov_type='uni_diag',
                 max_iter=100,
                 tol=0.000001,
                 eps=0.000001,
                 reg_covar=0.000001,
                 n_init=20,
                 _k_tol=1,
                 device="cpu",
                 verbose=False,
                 prop=None):
        super().__init__(k_range, criterion, cov_type, max_iter, tol, eps,
                         reg_covar, n_init, _k_tol, device, verbose)
        self.n = n
        self.d = d
        self.X = data_generator.sample_truth(n, d)
        self.Xna, self.M = self.ampute(self.X, patterns=patterns, prop=prop)
        self.model = self.fit(self.X, self.M)

    def mean(self):
        Xtilde = self.model.sample(self.n)[0]
        return Xtilde.mean(dim=0)

    def quantile(self, var_index=0, p=0.1):
        """
        Estimate the p-th quantile of the variable with index var_index
        """
        Xtilde = self.model.sample(self.n)[0]
        x = Xtilde[: var_index]
        return np.quantile(x, p)

    def estimate(self, func):
        Xtilde = self.model.sample(self.n)[0]
        return func(Xtilde)

    # mean, quantile, estimate unchanged from the previous version

Reference: K. V. Mardia (1970), Measures of Multivariate Skewness and Kurtosis with Applications, Biometrika, 57(3), 519–530. https://doi.org/10.1093/biomet/57.3.519

In [38]:

import torch


def mardia_skewness(X: torch.Tensor, ridge: float = 0.0) -> torch.Tensor:
    """
    Estimate Mardia's multivariate skewness.

    Args:
        X: Tensor of shape (n_samples, n_features).
        ridge: Optional diagonal regularization for covariance.

    Returns:
        Scalar tensor: estimated multivariate skewness.
    """
    n, p = X.shape

    # Center the data
    Xc = X - X.mean(dim=0, keepdim=True)

    # Maximum-likelihood-style sample covariance (denominator n)
    cov = (Xc.T @ Xc) / n

    # Optional regularization
    if ridge > 0:
        cov = cov + ridge * torch.eye(
            p, dtype=X.dtype, device=X.device
        )

    # Compute S^{-1} Xc^T without explicitly inverting S
    solved = torch.linalg.solve(cov, Xc.T)

    # Pairwise Mahalanobis inner products
    A = Xc @ solved

    # Mardia's multivariate skewness
    return A.pow(3).mean()



torch.manual_seed(42)

X = torch.randn(500, 3, dtype=torch.float64)

beta = mardia_skewness(X)

print("Mardia skewness:", beta.item())

Mardia skewness: 0.0710946339973778


In [12]:
dg = DataGenerator(distr='Logistic')

In [ ]:
n = 2000
d = 20
B = 100

In [17]:
means = torch.empty((B, d))
quantiles_X2 = torch.empty(B)

rng = np.random.default_rng(123)
# Sample incomplete missingness patterns once (1 = observed, 0 = missing).
patterns = AmputeMAR.sample_patterns(d, n, rng)   # once, outside the loop

print(patterns)

for i in range(B):
    pe = ParametricEstimation(data_generator=dg, n=n, d=d, patterns=patterns, k_range=(0,10))
    print((pe.M.sum(dim=1) == d).float().mean())
    means[i] = pe.mean()
    quantiles_X2[i] = pe.quantile(var_index=1, p=0.2).item()

[[0 1 1]
 [0 0 1]]
tensor(0.3175)
tensor(0.3390)


In [7]:
print('Mean estimates per batch: \n', means)

Mean estimates per batch: 
 tensor([[-0.0479, -0.0924, -0.0154],
        [ 0.1001,  0.0335, -0.0165]])


In [18]:
#print('Quantile 0.2 estimates of X2 per batch: \n', quantiles_X2)

q = quantiles_X2.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(4, 5))
ax.boxplot(q, widths=0.5, showmeans=True)
ax.scatter(np.ones_like(q), q, alpha=0.4, color="k", zorder=3)  # individual estimates

# optional: true value, if you know it
# ax.axhline(true_q, color="red", linestyle="--", label="True 0.2-quantile")
# ax.legend()

ax.set_xticks([1])
ax.set_xticklabels(["X2"])
ax.set_ylabel("Estimated 0.2-quantile")
ax.set_title(f"Quantile 0.2 estimates of X2 (B = {len(q)})")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined